<a href="https://colab.research.google.com/github/Balajivallepu/Nlp_projects/blob/main/NLP_PROJECT_%5E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install kagglehub nltk scikit-learn joblib pandas


In [10]:
import os
import re
import joblib
import pandas as pd
import kagglehub
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
path = kagglehub.dataset_download(
    "amananandrai/ag-news-classification-dataset"
)

print("Dataset Path:", path)

Using Colab cache for faster access to the 'ag-news-classification-dataset' dataset.
Dataset Path: /kaggle/input/ag-news-classification-dataset


In [12]:
train_path = os.path.join(path, "train.csv")
test_path = os.path.join(path, "test.csv")

train = pd.read_csv(
    train_path,
    header=None,
    names=["label", "title", "description"]
)

test = pd.read_csv(
    test_path,
    header=None,
    names=["label", "title", "description"]
)

print(train.head())

         label                                              title  \
0  Class Index                                              Title   
1            3  Wall St. Bears Claw Back Into the Black (Reuters)   
2            3  Carlyle Looks Toward Commercial Aerospace (Reu...   
3            3    Oil and Economy Cloud Stocks' Outlook (Reuters)   
4            3  Iraq Halts Oil Exports from Main Southern Pipe...   

                                         description  
0                                        Description  
1  Reuters - Short-sellers, Wall Street's dwindli...  
2  Reuters - Private investment firm Carlyle Grou...  
3  Reuters - Soaring crude prices plus worries\ab...  
4  Reuters - Authorities have halted oil export\f...  


In [13]:
print(train.shape)
print(test.shape)

train.info()

(120001, 3)
(7601, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120001 entries, 0 to 120000
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   label        120001 non-null  object
 1   title        120001 non-null  object
 2   description  120001 non-null  object
dtypes: object(3)
memory usage: 2.7+ MB


In [14]:
train["text"] = train["title"] + " " + train["description"]
test["text"] = test["title"] + " " + test["description"]

train.head()

,label,title,description,text
0,Class Index,Title,Description,Title Description
1,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Wall St. Bears Claw Back Into the Black (Reute...
2,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...
3,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Oil and Economy Cloud Stocks' Outlook (Reuters...
4,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,Iraq Halts Oil Exports from Main Southern Pipe...


In [15]:
label_map = {
    1: "World",
    2: "Sports",
    3: "Business",
    4: "Sci/Tech"
}

train["category"] = train["label"].map(label_map)
test["category"] = test["label"].map(label_map)

train.head()

,label,title,description,text,category
0,Class Index,Title,Description,Title Description,NaN
1,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Wall St. Bears Claw Back Into the Black (Reute...,NaN
2,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...,NaN
3,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Oil and Economy Cloud Stocks' Outlook (Reuters...,NaN
4,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,Iraq Halts Oil Exports from Main Southern Pipe...,NaN


In [16]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def preprocess(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z ]', ' ', text)

    words = text.split()

    words = [
        stemmer.stem(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

train["clean_text"] = train["text"].apply(preprocess)
test["clean_text"] = test["text"].apply(preprocess)

train[["clean_text"]].head()

,clean_text
0,titl descript
1,wall st bear claw back black reuter reuter sho...
2,carlyl look toward commerci aerospac reuter re...
3,oil economi cloud stock outlook reuter reuter ...
4,iraq halt oil export main southern pipelin reu...


In [17]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train = vectorizer.fit_transform(train["clean_text"])
X_test = vectorizer.transform(test["clean_text"])

y_train = train["label"]
y_test = test["label"]

In [18]:
model = MultinomialNB()

model.fit(X_train, y_train)

print("Model Training Completed")

Model Training Completed


In [19]:
predictions = model.predict(X_test)

print(predictions[:20])

['2' '3' '4' '4' '4' '4' '4' '4' '4' '4' '4' '4' '4' '4' '4' '2' '4' '4'
 '4' '4']


In [20]:
accuracy = accuracy_score(y_test, predictions)

print("Accuracy :", accuracy)

Accuracy : 0.8918563346928036


In [22]:
print(classification_report(
    y_test,
    predictions,
    labels=['1', '2', '3', '4'], # Explicitly specify labels as strings
    target_names=list(label_map.values())
))

              precision    recall  f1-score   support

       World       0.90      0.89      0.90      1900
      Sports       0.94      0.97      0.95      1900
    Business       0.86      0.84      0.85      1900
    Sci/Tech       0.86      0.87      0.86      1900

   micro avg       0.89      0.89      0.89      7600
   macro avg       0.89      0.89      0.89      7600
weighted avg       0.89      0.89      0.89      7600



In [23]:
print(confusion_matrix(y_test, predictions))

[[1689   69   94   48    0]
 [  27 1844   11   18    0]
 [  77   23 1600  200    0]
 [  81   27  146 1646    0]
 [   0    1    0    0    0]]


In [24]:
joblib.dump(model, "model.pkl")
joblib.dump(vectorizer, "tfidf.pkl")

print("Model Saved Successfully")

Model Saved Successfully


In [ ]:
while True:

    news = input("Enter News (exit to quit): ")

    if news.lower() == "exit":
        break

    clean = preprocess(news)

    vector = vectorizer.transform([clean])

    prediction = model.predict(vector)[0]

    print("Predicted Category:", label_map[int(prediction)])

Enter News (exit to quit): India defeated Australia by 5 wickets in the ICC World Cup final.
Predicted Category: Sports
Enter News (exit to quit): India defeated Australia by 5 wickets in the ICC World Cup final.
Predicted Category: Sports
